In [1]:
from pathlib import Path

import pandas as pd
from sklearn.metrics import classification_report

BASE_DIR = Path.cwd().parent

Define Analytical Pipeline Functions

In [2]:
"""Function to Generate Hallucination Rate and Classification Report of the Given Result Dataframe"""

def generate_classification_report (model_result:pd.DataFrame, prompt_type:str):
    classification_df = model_result.copy()
    
    print(f"====== Classification Analysis: {prompt_type} ======")
    
    # Hallucinated labels percentage
    hallucinated_percentage = (classification_df["status"]!="Valid").sum() / len(classification_df) * 100
    print(f"Valid Labels Count: {classification_df[classification_df['status']=='Valid'].shape[0]} ; Invalid Labels Count: {classification_df[classification_df['status']!='Valid'].shape[0]}")
    print(f"Hallucinated Labels Percentage: {hallucinated_percentage:.2f}%")
    
    # Remove invalid labels for analysis
    classification_df = classification_df[classification_df["status"]=="Valid"].copy()
    
    # sort data based on true labels for proper CR and CM generation
    labels = sorted(classification_df["True_label"].unique())
    
    print(classification_report(classification_df["True_label"],classification_df["predicted_label"], labels=labels, target_names=labels))


In [3]:
"""Function to Extract and Show Inference Efficiency Results"""

def extract_time(df:pd.DataFrame, model:str): 
    print(f"============= Inference Efficiency Analysis: {model} ============")
    if model == "GPT-5.6-luna":
        time_df = df[["input_tokens", "output_tokens"]]
        print(
                "Input efficiency:\n",
                f"  - Total input tokens = {time_df["input_tokens"].sum():.0f}\n",
                f"  - Average input tokens = {time_df["input_tokens"].mean():.0f}\n",
                
                "Output efficiency:\n",
                f"  - Total output tokens = {time_df["output_tokens"].sum():.0f}\n",
                f"  - Average output tokens = {time_df["output_tokens"].mean():.0f}\n",
        )
    
    else:
        time_df = df[["input_tokens", "prompt_eval_duration","output_tokens", "eval_duration", "total_duration"]]
        
        print(
            "Input efficiency:\n",
            f"  - Total input tokens = {time_df["input_tokens"].sum():.0f}\n",
            f"  - Average input tokens = {time_df["input_tokens"].mean():.0f}\n\n",
            
            "Output efficiency:\n",
            f"  - Total output tokens = {time_df["output_tokens"].sum():.0f}\n",
            f"  - Average output tokens = {time_df["output_tokens"].mean():.0f}\n\n",
            
            "Inference efficiency:\n",
            f"  - Total inference time = {time_df["total_duration"].sum():.2f}\n",
            f"  - Average inference time = {time_df["total_duration"].mean():.2f}\n\n",
        )

    

Example Analysis

In [5]:
""" Load Results Dataframes for Analysis and Define Prompt Type Used for Analysis """

# Amend the file names based on your savefiles, below are just examples of the files used for analysis
mistral = pd.read_csv(f"{BASE_DIR}/reports/open-source_models/json_RAG_tfidf_mistral_timestamp.csv")
qwen = pd.read_csv(f"{BASE_DIR}/reports/open-source_models/json_RAG_tfidf_qwen3.5-9b_timestamp.csv")
llama = pd.read_csv(f"{BASE_DIR}/reports/open-source_models/json_RAG_tfidf_llama3-8b_timestamp.csv")
gemma = pd.read_csv(f"{BASE_DIR}/reports/open-source_models/json_RAG_tfidf_gemma3-4b_timestamp.csv")
# gpt_luna = pd.read_csv(f"{BASE_DIR}/reports/gpt_batch/batch_output/gpt6_json_tfidf_parsed_output_file_batch_batch_id_timestamp.csv")

models = {
    "Mistral-7B": mistral,
    "Qwen3.5-9B": qwen,
    "Llama3-8B": llama,
    "Gemma3-4B": gemma, 
    # "GPT-5.6-Luna": gpt_luna
}

prompt_type = "RAG Prompt (Json Format)" # amend based on prompt type used for the analysis 

In [ ]:
""" Report Generation Cell """

# sample analysis run
generate_classification_report(mistral, f"Mistral-7B {prompt_type}")
extract_time(mistral, "Mistral-7B")

# Run analysis for all models (uncomment the code below to run for all models)
# for model, result in models.items():
    # generate_classification_report(result, f"{model} {prompt_type}")
    # print("\n")
    # extract_time(result, model)
    # print("\n")